# 🥇 Camada Gold — Analytics SISCAN / 10ª CRS

Transforma as tabelas consolidadas da **Camada Silver** em tabelas fato analíticas no schema `gold`, calculando as métricas exigidas pelas questões de pesquisa do TCC (Q1–Q5): distribuição espacial e evolução temporal, efeito do Outubro Rosa, conformidade da faixa etária, adequabilidade das amostras e proporção de exames alterados.

**Princípios:** preservação de nulos reais (sem `coalesce(x, 0)`), rastreabilidade (`_silver_source`, `_gold_processed_time`) e validação estrita (PK única, sem nulos em colunas obrigatórias) **antes** de cada gravação.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Catálogo e schemas
CATALOG = "`tcc-analise-siscan-10crs`"
SCHEMA_SILVER = "silver"
SCHEMA_GOLD = "gold"

# 1. Escopo Temporal
ANO_INICIO = 2020
ANO_FIM = 2025

# Garantir schema gold
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_GOLD}")
print(f"✅ Schema {CATALOG}.{SCHEMA_GOLD} garantido.")


def validar_e_gravar(df, tabela_gold, pk_cols, obrigatorio_cols):
    """Valida PK única e ausência de nulos em colunas obrigatórias ANTES de gravar na Gold."""
    nome = f"{CATALOG}.{SCHEMA_GOLD}.{tabela_gold}"
    n = df.count()

    # 1. Chave primária única
    dups = df.groupBy(pk_cols).count().filter(F.col("count") > 1).count()
    assert dups == 0, f"❌ {tabela_gold}: {dups} grupos de PK duplicada em {pk_cols}"

    # 2. Nulos em colunas obrigatórias (dimensões — métricas podem ser nulas por decisão de negócio)
    for c in obrigatorio_cols:
        nulos = df.filter(F.col(c).isNull()).count()
        assert nulos == 0, f"❌ {tabela_gold}: {nulos} nulos na coluna obrigatória '{c}'"

    # 3. Gravar somente se as validações passarem
    (
        df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(nome)
    )
    print(f"✅ {tabela_gold}: {n} linhas gravadas (PK {pk_cols} validada).")
    return n

## 1. GOLD: `fato_exames_temporais_territoriais` (Q1 + Q2)

**Granularidade:** Município × Mês/Ano (`dt_competencia`).

**Métricas (Especificação do Problema Analítico):**
* Quantidade de exames ($Q_{m,t}$)
* Participação Territorial (%) $= \frac{\text{Exames do Município}}{\text{Total 10ª CRS no Período}} \times 100$
* Variação Absoluta $= Q_t - Q_{t-1}$
* Variação Percentual (%) $= \frac{Q_t - Q_{t-1}}{Q_{t-1}} \times 100$

> **Nulos reais preservados:** quando o mês anterior não existe ou a quantidade é ausente na Silver, as variações ficam `NULL` — não são fabricados zeros falsos (Ponto 5 das diretrizes).

In [0]:
# ---------------------------------------------------------------------------
# 1. GOLD: fato_exames_temporais_territoriais (Q1 + Q2)
# ---------------------------------------------------------------------------
df_silver_exames = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.exames_geral")

# Total da regional no mesmo mês/ano (janela por competência)
win_regional = Window.partitionBy("dt_competencia")
# Quantidade do mês anterior (Q_{t-1}) por município
win_lag_muni = Window.partitionBy("codigo_municipio").orderBy("dt_competencia")

df_q1_q2 = (
    df_silver_exames
    .withColumn("cd_ibge", F.lpad(F.col("codigo_municipio").cast("string"), 6, "0"))
    .withColumn("_silver_source", F.lit(f"{CATALOG}.{SCHEMA_SILVER}.exames_geral"))
    .withColumn("_gold_processed_time", F.current_timestamp())
    .withColumn("total_exames_10crs", F.sum("quantidade").over(win_regional))
    .withColumn(
        "participacao_territorial_pct",
        F.when(F.col("total_exames_10crs") > 0,
               F.round((F.col("quantidade") / F.col("total_exames_10crs")) * 100, 2)),
    )
    # Correção (31/out/2026): o lag só vale se a linha anterior for o mês CALENDRÁRIO
    # imediatamente precedente. Sem essa checagem, set/2022 compararia com jul/2022
    # (Δ de 2 meses) porque ago/2022 não existe na fonte (TABNET omite a coluna —
    # 0 exames na região inteira). Se o mês anterior calendário não existir,
    # quantidade_mes_anterior e as variações ficam NULL (nulo real preservado).
    .withColumn("_dt_lag", F.lag("dt_competencia", 1).over(win_lag_muni))
    .withColumn("_qtde_lag", F.lag("quantidade", 1).over(win_lag_muni))
    .withColumn(
        "quantidade_mes_anterior",
        F.when(F.add_months(F.col("_dt_lag"), 1) == F.col("dt_competencia"), F.col("_qtde_lag")),
    )
    .withColumn("variacao_absoluta", F.col("quantidade") - F.col("quantidade_mes_anterior"))
    .withColumn(
        "variacao_percentual_pct",
        F.when(F.col("quantidade_mes_anterior") > 0,
               F.round(((F.col("quantidade") - F.col("quantidade_mes_anterior"))
                        / F.col("quantidade_mes_anterior")) * 100, 2)),
    )
    .select(
        "codigo_municipio", "cd_ibge", "municipio", "ano", "mes", "dt_competencia",
        "quantidade", "total_exames_10crs", "participacao_territorial_pct",
        "quantidade_mes_anterior", "variacao_absoluta", "variacao_percentual_pct",
        "_silver_source", "_gold_processed_time",
    )
)

validar_e_gravar(
    df_q1_q2,
    "fato_exames_temporais_territoriais",
    pk_cols=["codigo_municipio", "dt_competencia"],
    obrigatorio_cols=["codigo_municipio", "municipio", "ano", "mes", "dt_competencia"],
)

display(df_q1_q2.orderBy("codigo_municipio", "dt_competencia").limit(12))

## 2. GOLD: `fato_analise_outubro_rosa` (Q2 — Campanha)

**Granularidade:** Município × Ano × Mês da Campanha (Out/Nov/Dez).

**Fórmula (Especificação):** $\text{Variação Campanha (\%)} = \frac{\text{Exames Mês Associado} - \text{Exames Setembro}}{\text{Exames Setembro}} \times 100$

* Setembro é o período de referência do mesmo ano/município.
* Agregação prévia por (`codigo_municipio`, `ano`, `mes`) garante o join 1:1 mesmo se a Silver contiver múltiplas linhas por competência.

In [0]:
# ---------------------------------------------------------------------------
# 2. GOLD: fato_analise_outubro_rosa (Q2 — comparação com Setembro)
# ---------------------------------------------------------------------------
df_campanha_base = df_silver_exames.filter(F.col("mes").isin([9, 10, 11, 12]))

# Referência: exames de Setembro do mesmo ano/município (agregado → join 1:1)
df_setembro = (
    df_campanha_base
    .filter(F.col("mes") == 9)
    .groupBy("codigo_municipio", "ano")
    .agg(F.sum("quantidade").alias("exames_referencia_setembro"))
)

df_q2_campanha = (
    df_campanha_base
    .filter(F.col("mes").isin([10, 11, 12]))
    .groupBy("codigo_municipio", "municipio", "ano", "mes", "dt_competencia")
    .agg(F.sum("quantidade").alias("exames_mes"))
    .join(df_setembro, on=["codigo_municipio", "ano"], how="left")
    .withColumn(
        "ds_periodo_campanha",
        F.when(F.col("mes") == 10, F.lit("Mês da Campanha (Outubro)"))
         .when(F.col("mes") == 11, F.lit("Mês Subsequente 1 (Novembro)"))
         .otherwise(F.lit("Mês Subsequente 2 (Dezembro)")),
    )
    .withColumn(
        "variacao_campanha_pct",
        F.when(F.col("exames_referencia_setembro") > 0,
               F.round(((F.col("exames_mes") - F.col("exames_referencia_setembro"))
                        / F.col("exames_referencia_setembro")) * 100, 2)),
    )
    .withColumn("cd_ibge", F.lpad(F.col("codigo_municipio").cast("string"), 6, "0"))
    .withColumn("_silver_source", F.lit(f"{CATALOG}.{SCHEMA_SILVER}.exames_geral"))
    .withColumn("_gold_processed_time", F.current_timestamp())
    .select(
        "codigo_municipio", "cd_ibge", "municipio", "ano", "mes", "dt_competencia",
        "ds_periodo_campanha", "exames_mes", "exames_referencia_setembro",
        "variacao_campanha_pct", "_silver_source", "_gold_processed_time",
    )
)

validar_e_gravar(
    df_q2_campanha,
    "fato_analise_outubro_rosa",
    pk_cols=["codigo_municipio", "ano", "mes"],
    obrigatorio_cols=["codigo_municipio", "municipio", "ano", "mes", "ds_periodo_campanha"],
)

display(df_q2_campanha.orderBy("codigo_municipio", "ano", "mes").limit(12))

## 3. GOLD: `fato_conformidade_faixa_etaria` (Q3)

**Granularidade:** Município × Ano × **Faixa Etária detalhada (5 em 5 anos)** — cada linha mantém também o `grupo_etario_diretriz` (3 agrupamentos macro) e as métricas do grupo (`quantidade_grupo`, `proporcao_grupo_pct`), permitindo filtrar por faixa quinquenal **ou** por grupo macro no dashboard.

**Categorias exigidas pelo TCC:**
* `<25 anos` — Sobrerrastreamento
* `25 a 64 anos` — População-Alvo (Ministério da Saúde)
* `≥65 anos` — Sub-rastreamento

> **Classificação robusta:** a Silver armazena faixas como `"10 a 14 anos"`, `"Até 9 anos"`, `"Acima de 79 anos"`. A classificação usa o limite inferior extraído numericamente (`regexp_extract`) + `fl_populacao_alvo`, cobrindo todos os 16 valores reais da Silver.

In [0]:
# ---------------------------------------------------------------------------
# 3. GOLD: fato_conformidade_faixa_etaria (Q3)
# ---------------------------------------------------------------------------
df_faixa = (
    spark.table(f"{CATALOG}.{SCHEMA_SILVER}.faixa_etaria")
    # Correção (31/out/2026): exclui a faixa 'Ignorado' (2021) — sem informação etária,
    # não se enquadra em nenhum dos 3 grupos da diretriz (sem o filtro cairia no
    # otherwise como '65 anos ou mais'). 'Até 9 anos' (2024/2025) permanece: idade
    # informada → classifica corretamente como sobrerastreamento (<25 anos).
    .filter(F.col("faixa_etaria") != "Ignorado")
)

# Classificação em 3 grupos da diretriz — extrai o limite inferior numérico da faixa
df_faixa_grupo = df_faixa.withColumn(
    "idade_limite_inferior",
    F.expr("try_cast(regexp_extract(faixa_etaria, '([0-9]+)', 1) as int)"),
).withColumn(
    "grupo_etario_diretriz",
    F.when(F.col("fl_populacao_alvo"), F.lit("25 a 64 anos (População-Alvo)"))
     .when(F.col("idade_limite_inferior") < 25, F.lit("<25 anos (Sobrerrastreamento)"))
     .otherwise(F.lit("65 anos ou mais (Sub-rastreamento)")),
)

win_muni_ano = Window.partitionBy("codigo_municipio", "ano")
win_grupo = Window.partitionBy("codigo_municipio", "ano", "grupo_etario_diretriz")

df_q3 = (
    df_faixa_grupo
    .groupBy("codigo_municipio", "municipio", "ano", "faixa_etaria", "grupo_etario_diretriz")
    .agg(F.sum("quantidade").alias("quantidade_exames"))
    # Total do município/ano (todas as faixas com informação etária)
    .withColumn("total_exames_muni_ano", F.sum("quantidade_exames").over(win_muni_ano))
    # Proporção da FAIXA DETALHADA (5 em 5 anos) dentro do município/ano
    .withColumn(
        "proporcao_pct",
        F.when(F.col("total_exames_muni_ano") > 0,
               F.round((F.col("quantidade_exames") / F.col("total_exames_muni_ano")) * 100, 2)),
    )
    # Métricas do GRUPO MACRO (3 agrupamentos da diretriz), repetidas nas faixas do grupo
    .withColumn("quantidade_grupo", F.sum("quantidade_exames").over(win_grupo))
    .withColumn(
        "proporcao_grupo_pct",
        F.when(F.col("total_exames_muni_ano") > 0,
               F.round((F.col("quantidade_grupo") / F.col("total_exames_muni_ano")) * 100, 2)),
    )
    .withColumn("cd_ibge", F.lpad(F.col("codigo_municipio").cast("string"), 6, "0"))
    .withColumn("_silver_source", F.lit(f"{CATALOG}.{SCHEMA_SILVER}.faixa_etaria"))
    .withColumn("_gold_processed_time", F.current_timestamp())
    .select(
        "codigo_municipio", "cd_ibge", "municipio", "ano", "faixa_etaria",
        "grupo_etario_diretriz", "quantidade_exames", "total_exames_muni_ano",
        "proporcao_pct", "quantidade_grupo", "proporcao_grupo_pct",
        "_silver_source", "_gold_processed_time",
    )
)

validar_e_gravar(
    df_q3,
    "fato_conformidade_faixa_etaria",
    pk_cols=["codigo_municipio", "ano", "faixa_etaria"],
    obrigatorio_cols=["codigo_municipio", "municipio", "ano", "faixa_etaria", "grupo_etario_diretriz"],
)

display(df_q3.orderBy("codigo_municipio", "ano", "faixa_etaria").limit(20))

### 3b. GOLD: `fato_indicador_populacao_alvo` (Q3 — KPI isolado)

**Granularidade:** Município × Ano (1 linha por município/ano — ideal para cartões KPI no Power BI / AI-BI Dashboard).

**Fórmula (Especificação):** $\text{Proporção na população-alvo (\%)} = \frac{\text{Exames de 25 a 64 anos}}{\text{Total de exames com informação etária}} \times 100$

Além do KPI principal (25–64 anos), a tabela traz as proporções de sobrerrastreamento (<25) e sub-rastreamento (≥65) em colunas separadas na mesma linha.

In [0]:
# ---------------------------------------------------------------------------
# 3b. GOLD: fato_indicador_populacao_alvo (Q3 — KPI da população-alvo)
# ---------------------------------------------------------------------------
GRUPO_ALVO = "25 a 64 anos (População-Alvo)"
GRUPO_SOBRE = "<25 anos (Sobrerrastreamento)"
GRUPO_SUB = "65 anos ou mais (Sub-rastreamento)"

df_indicador_alvo = (
    df_q3
    .groupBy("codigo_municipio", "municipio", "ano")
    .agg(
        F.max("total_exames_muni_ano").alias("total_exames_informacao_etaria"),
        F.max(F.when(F.col("grupo_etario_diretriz") == GRUPO_ALVO, F.col("quantidade_grupo"))).alias("exames_populacao_alvo"),
        F.max(F.when(F.col("grupo_etario_diretriz") == GRUPO_SOBRE, F.col("quantidade_grupo"))).alias("exames_sobrerastreamento"),
        F.max(F.when(F.col("grupo_etario_diretriz") == GRUPO_SUB, F.col("quantidade_grupo"))).alias("exames_subrastreamento"),
    )
    .withColumn(
        "proporcao_populacao_alvo_pct",
        F.when(F.col("total_exames_informacao_etaria") > 0,
               F.round((F.col("exames_populacao_alvo") / F.col("total_exames_informacao_etaria")) * 100, 2)),
    )
    .withColumn(
        "proporcao_sobrerastreamento_pct",
        F.when(F.col("total_exames_informacao_etaria") > 0,
               F.round((F.col("exames_sobrerastreamento") / F.col("total_exames_informacao_etaria")) * 100, 2)),
    )
    .withColumn(
        "proporcao_subrastreamento_pct",
        F.when(F.col("total_exames_informacao_etaria") > 0,
               F.round((F.col("exames_subrastreamento") / F.col("total_exames_informacao_etaria")) * 100, 2)),
    )
    .withColumn("cd_ibge", F.lpad(F.col("codigo_municipio").cast("string"), 6, "0"))
    .withColumn("_silver_source", F.lit(f"{CATALOG}.{SCHEMA_SILVER}.faixa_etaria"))
    .withColumn("_gold_processed_time", F.current_timestamp())
    .select(
        "codigo_municipio", "cd_ibge", "municipio", "ano",
        "exames_populacao_alvo", "exames_sobrerastreamento", "exames_subrastreamento",
        "total_exames_informacao_etaria",
        "proporcao_populacao_alvo_pct", "proporcao_sobrerastreamento_pct",
        "proporcao_subrastreamento_pct", "_silver_source", "_gold_processed_time",
    )
)

validar_e_gravar(
    df_indicador_alvo,
    "fato_indicador_populacao_alvo",
    pk_cols=["codigo_municipio", "ano"],
    obrigatorio_cols=["codigo_municipio", "municipio", "ano"],
)

display(df_indicador_alvo.orderBy("codigo_municipio", "ano"))

### 3c. GOLD: `fato_evolucao_cobertura_municipios` (Q3 — Razão Anual de Exames)

**Granularidade:** Município × Ano (2020–2025 → 66 linhas).

**Fórmula (Especificação):**

$\text{Razão Anual de Exames do Ano } t (\%) = \left( \dfrac{\text{Exames realizados na faixa 25--64 anos no ano } t}{\text{População Feminina Alvo IBGE 2022 (25--64 anos)}} \right) \times 100$

* **Numerador:** exames com `fl_populacao_alvo = True` na Silver `faixa_etaria` (faixas de 25 a 64 anos), agregados por município e ano.
* **Denominador:** teto populacional **fixo** do Censo IBGE 2022 — mulheres de 25 a 64 anos de `gold.dim_populacao_municipios` (11 municípios da 10ª CRS, total 123.093) — constante para todos os anos do período.
* **Referência exploratória:** linha de 33,0% ao ano para comparação de produção relativa à população. Não corresponde a comprovação de cobertura individual nem ao atingimento de meta de mulheres únicas rastreadas. O campo legado `st_meta_ans` está mantido para compatibilidade; seus rótulos não devem ser interpretados como cobertura efetiva.
* **Consumo:** página **Q3 — Faixa etária** do app Streamlit (`src/app/pages/03_faixa_etaria.py`, seção "Razão anual de exames").

**Interpretação e limitação:** o numerador soma **procedimentos registrados**, não o número de mulheres distintas nem necessariamente o primeiro procedimento. Portanto, esta métrica expressa a produção anual de exames por 100 mulheres da população-alvo (denominador fixo IBGE 2022), **não a cobertura populacional efetiva**. Nome canônico da métrica: `razao_anual_exames_pct`. O campo `taxa_cobertura_pct` permanece somente como alias retrocompatível para consumidores antigos.


In [0]:
# ---------------------------------------------------------------------------
# 3c. GOLD: fato_evolucao_cobertura_municipios (Q3 — Razão Anual de Exames)
# ---------------------------------------------------------------------------
# Numerador: exames da população-alvo (25–64 anos) por município e ano
df_exames_alvo = (
    spark.table(f"{CATALOG}.{SCHEMA_SILVER}.faixa_etaria")
    .filter(F.col("fl_populacao_alvo") == True)
    .groupBy("codigo_municipio", "municipio", "ano")
    .agg(F.sum("quantidade").alias("total_exames_25_64"))
)

# Denominador: teto populacional fixo (Censo IBGE 2022) — mulheres 25–64 anos
df_pop_alvo = (
    spark.table(f"{CATALOG}.{SCHEMA_GOLD}.dim_populacao_municipios")
    .select(
        F.col("cd_ibge").cast("int").alias("codigo_municipio"),
        F.col("mulheres_25_64_anos"),
    )
)

df_evolucao_cobertura = (
    df_exames_alvo
    .join(df_pop_alvo, "codigo_municipio", "inner")
    # Razão anual de exames (%): soma de procedimentos / população feminina fixa.
    # NÃO mede mulheres distintas examinadas nem primeiro procedimento por mulher.
    .withColumn(
        "razao_anual_exames_pct",
        F.when(
            F.col("mulheres_25_64_anos") > 0,
            F.round((F.col("total_exames_25_64") / F.col("mulheres_25_64_anos")) * 100, 2),
        ),
    )
    # Alias técnico para compatibilidade com consumidores da Gold existentes.
    .withColumn("taxa_cobertura_pct", F.col("razao_anual_exames_pct"))
    # Referência exploratória de 33%; não significa cobertura de mulheres únicas
    .withColumn("meta_ans_pct", F.lit(33.0))
    # Classificação legada de produção relativa à população (não cobertura individual)
    .withColumn(
        "st_meta_ans",
        F.when(F.col("razao_anual_exames_pct") >= 33.0, F.lit("🟢 Razão ≥33%"))
         .when(F.col("razao_anual_exames_pct") >= 3.0, F.lit("🟡 Razão de 3% a <33%"))
         .otherwise(F.lit("🔴 Razão <3%")),
    )
    .withColumn(
        "_silver_source",
        F.lit(f"{CATALOG}.{SCHEMA_SILVER}.faixa_etaria + {CATALOG}.{SCHEMA_GOLD}.dim_populacao_municipios"),
    )
    .withColumn("_gold_processed_time", F.current_timestamp())
)

# Guardas: o inner join deve preservar os 11 municípios × 6 anos (66 linhas)
assert df_evolucao_cobertura.count() == 66, \
    f"Esperado 66 linhas (11 municípios × 6 anos); obtido {df_evolucao_cobertura.count()}"
assert df_evolucao_cobertura.select("codigo_municipio").distinct().count() == 11, \
    "Inner join com a dimensão populacional perdeu municípios da 10ª CRS"

validar_e_gravar(
    df_evolucao_cobertura,
    "fato_evolucao_cobertura_municipios",
    pk_cols=["codigo_municipio", "ano"],
    obrigatorio_cols=["codigo_municipio", "municipio", "ano", "total_exames_25_64",
                      "mulheres_25_64_anos", "razao_anual_exames_pct"],
)

display(df_evolucao_cobertura.orderBy("codigo_municipio", "ano"))

### 3d. GOLD: `dim_meta_ans_municipios` (Q3 — Meta Anual de Exames ANS/MS)

**Granularidade:** Município (11 linhas — a meta NÃO varia por ano, pois o denominador é o teto populacional fixo do Censo 2022).

**Fórmula (Especificação):**

$\text{Meta Anual de Exames (Cidade)} = \text{População Feminina Alvo (25 a 64 anos) [IBGE 2022]} \times 0{,}33$

* A diretriz técnica ANS/MS fixa a meta anual em $\ge 33{,}0\%$ da população feminina de 25 a 64 anos. Atingi-la em 1 ano garante 100% das mulheres elegíveis rastreadas ao longo de um triênio (rotina epidemiológica de 1 exame a cada 3 anos).
* **Volume absoluto:** `meta_ans_exames_anual` = população × 0,33, arredondado para exames inteiros (`F.round`). **Percentual:** `meta_ans_pct` = 33,0.
* **Consumo:** página **Q3 — Faixa etária** do app Streamlit (comparação da razão de exames com patamar de referência).

In [0]:
# ---------------------------------------------------------------------------
# 3d. GOLD: dim_meta_ans_municipios (Q3 — Meta Anual de Exames ANS/MS)
# ---------------------------------------------------------------------------
df_meta_ans = (
    spark.table(f"{CATALOG}.{SCHEMA_GOLD}.dim_populacao_municipios")
    # Meta Anual de Exames = População Feminina Alvo (25–64) [IBGE 2022] × 0,33
    .withColumn(
        "meta_ans_exames_anual",
        F.round(F.col("mulheres_25_64_anos") * 0.33, 0).cast("long"),
    )
    .withColumn("meta_ans_pct", F.lit(33.0))
    .withColumn("codigo_municipio", F.col("cd_ibge").cast("int"))
    .withColumn("_silver_source", F.lit(f"{CATALOG}.{SCHEMA_GOLD}.dim_populacao_municipios (Censo IBGE 2022)"))
    .withColumn("_gold_processed_time", F.current_timestamp())
    .select(
        "codigo_municipio", "cd_ibge", "municipio", "mulheres_25_64_anos",
        "meta_ans_exames_anual", "meta_ans_pct", "_silver_source", "_gold_processed_time",
    )
)

# Guarda: exatamente os 11 municípios da 10ª CRS
assert df_meta_ans.count() == 11, \
    f"Meta ANS deve ter 11 municípios da 10ª CRS; obtido {df_meta_ans.count()}"

validar_e_gravar(
    df_meta_ans,
    "dim_meta_ans_municipios",
    pk_cols=["codigo_municipio"],
    obrigatorio_cols=["codigo_municipio", "municipio", "mulheres_25_64_anos", "meta_ans_exames_anual"],
)

display(df_meta_ans.orderBy("codigo_municipio"))

## 4. GOLD: `fato_adequabilidade_amostras` (Q4)

**Granularidade:** Município × Ano × Categoria de Adequabilidade (Satisfatória / Insatisfatória / Rejeitada).

**Fórmula (Especificação):** $\text{Amostras Insatisfatórias (\%)} = \frac{\text{Amostras Insatisfatórias}}{\text{Total de Amostras com Adequabilidade}} \times 100$

In [0]:
# ---------------------------------------------------------------------------
# 4. GOLD: fato_adequabilidade_amostras (Q4)
# ---------------------------------------------------------------------------
df_adeq = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.adequabilidade")

win_adeq_total = Window.partitionBy("codigo_municipio", "ano")

df_q4 = (
    df_adeq
    .groupBy("codigo_municipio", "municipio", "ano", "categoria_adequabilidade")
    .agg(F.sum("quantidade").alias("quantidade_amostras"))
    .withColumn("total_amostras_muni_ano", F.sum("quantidade_amostras").over(win_adeq_total))
    .withColumn(
        "proporcao_adequabilidade_pct",
        F.when(F.col("total_amostras_muni_ano") > 0,
               F.round((F.col("quantidade_amostras") / F.col("total_amostras_muni_ano")) * 100, 2)),
    )
    .withColumn("cd_ibge", F.lpad(F.col("codigo_municipio").cast("string"), 6, "0"))
    .withColumn("_silver_source", F.lit(f"{CATALOG}.{SCHEMA_SILVER}.adequabilidade"))
    .withColumn("_gold_processed_time", F.current_timestamp())
    .select(
        "codigo_municipio", "cd_ibge", "municipio", "ano", "categoria_adequabilidade",
        "quantidade_amostras", "total_amostras_muni_ano", "proporcao_adequabilidade_pct",
        "_silver_source", "_gold_processed_time",
    )
)

validar_e_gravar(
    df_q4,
    "fato_adequabilidade_amostras",
    pk_cols=["codigo_municipio", "ano", "categoria_adequabilidade"],
    obrigatorio_cols=["codigo_municipio", "municipio", "ano", "categoria_adequabilidade"],
)

display(df_q4.orderBy("codigo_municipio", "ano", "categoria_adequabilidade"))

## 5. GOLD: `fato_exames_alterados` (Q5)

**Granularidade:** Município × Ano × Faixa Etária.

**Fórmula (Especificação):** $\text{Exames Alterados (\%)} = \frac{\text{Exames Alterados (colo)}}{\text{Total de Exames da Faixa Etária}} \times 100$

* Numerador: `silver.resultados_citopatologicos` (`grupo_resultado` = exames citopatológicos de colo **alterados** por faixa etária).
* Denominador: `silver.faixa_etaria` **menos amostras insatisfatórias/rejeitadas** (`silver.adequabilidade`) — apenas exames analisáveis, que permitem laudo diagnóstico definitivo.
* Faixas sem exames no denominador ficam com proporção `NULL` (sem divisão por zero fabricada).
* **Correção (31/out/2026):** a junção com o denominador é `FULL OUTER` — faixas do relatório de faixa etária sem registro de alterados (colunas zeradas omitidas pelo TABNET, ex.: "10 a 14" em 2024/25) entram com `quantidade_alterados = 0`, e o denominador fecha exatamente com o total de exames da regional (131.058).

> **Nota metodológica (03/nov/2026 — exclusão de insatisfatórias/rejeitadas do denominador):** amostras **rejeitadas** não são analisadas e amostras **insatisfatórias** não permitem laudo diagnóstico definitivo — logo o denominador agora considera apenas exames analisáveis. Como o SISCAN não cruza adequabilidade × faixa etária (o relatório por idade não segmenta por categoria e o de adequabilidade não segmenta por idade), as exclusões por município/ano (`silver.adequabilidade`, categorias Insatisfatória + Rejeitada) são **distribuídas proporcionalmente** ao volume de exames de cada faixa do mesmo município/ano, com alocação cumulativa inteira exata (a soma por município/ano fecha exatamente com o total excluído; colunas de auditoria `qtd_excluidas_muni_ano` e `excluidas_faixa`). O pressuposto de que as exclusões seguem o mesmo padrão etário dos exames deve ser declarado nas limitações do TCC.

> **Atualização (07/nov/2026 — grupo etário em 3 categorias):** nova coluna `grupo_etario_3` classifica cada faixa em **`<= 24 anos`**, **`25 a 64 anos`** ou **`>= 65 anos`** (`Ignorado` permanece categoria própria). A granularidade da tabela **não muda** (município × ano × faixa) — `faixa_etaria` é preservada para as análises detalhadas da Seção 8; a matriz territorial de proporção de alterados pode ser agregada nos 3 grupos com **taxa ponderada** `soma(quantidade_alterados) ÷ soma(total_exames_validos)` (nunca média das `proporcao_alterados_pct`). O agrupamento equivale às fronteiras do `grupo_etario_diretriz` de `fato_conformidade_faixa_etaria` (<25 / 25–64 / 65+), com rótulos próprios.

In [0]:
# ---------------------------------------------------------------------------
# 5. GOLD: fato_exames_alterados (Q5)
# ---------------------------------------------------------------------------
df_resultados = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.resultados_citopatologicos")
df_exames_totais = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.faixa_etaria")
df_adequabilidade = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.adequabilidade")

# Numerador: exames alterados por município/ano/faixa (agregado → join 1:1)
df_alterados = (
    df_resultados
    .groupBy("codigo_municipio", "municipio", "ano", "faixa_etaria")
    .agg(F.sum("quantidade").alias("quantidade_alterados"))
)

# Denominador: total de exames por município/ano/faixa
# (municipio entra no groupBy para viabilizar o full_outer abaixo — coluna obrigatória)
df_denominador = (
    df_exames_totais
    .groupBy("codigo_municipio", "municipio", "ano", "faixa_etaria")
    .agg(F.sum("quantidade").alias("total_exames_faixa"))
)

# Correção (03/nov/2026): EXCLUSÃO de amostras Insatisfatórias e Rejeitadas do
# denominador — rejeitadas não são analisadas e insatisfatórias não permitem
# laudo diagnóstico definitivo; apenas exames analisáveis compõem a base.
df_exclusoes = (
    df_adequabilidade
    .filter(F.col("categoria_adequabilidade").isin("Insatisfatoria", "Rejeitada"))
    .groupBy("codigo_municipio", "ano")
    .agg(F.sum("quantidade").alias("qtd_excluidas_muni_ano"))
)

# O SISCAN não cruza adequabilidade × faixa etária, então as exclusões de cada
# município/ano são distribuídas PROPORCIONALMENTE ao volume de exames de cada
# faixa. Alocação cumulativa com resto maior (floor da soma cumulativa das quotas):
# resultado inteiro e soma EXATA por município/ano — sem drift de arredondamento.
win_faixa = Window.partitionBy("codigo_municipio", "ano").orderBy("faixa_etaria")
df_denominador = (
    df_denominador
    .withColumn("total_exames_muni_ano",
        F.sum("total_exames_faixa").over(win_faixa.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)))
    .join(df_exclusoes, ["codigo_municipio", "ano"], how="left")
    .withColumn("qtd_excluidas_muni_ano", F.coalesce(F.col("qtd_excluidas_muni_ano"), F.lit(0)))
    # Aritmética INTEIRA exata (sem ponto flutuante — o floor de soma cumulativa em
    # double perdia 1 unidade por erro de representação): quota_i = faixa_i × E;
    # alocação_k = (cum_k div T) − (cum_{k−1} div T) com divisão inteira `div`.
    # A última linha fecha em E exatamente (cum_N = T×E → T×E div T = E).
    .withColumn("_quota_num", F.col("total_exames_faixa") * F.col("qtd_excluidas_muni_ano"))
    .withColumn("_cum_hi", F.sum("_quota_num").over(win_faixa.rowsBetween(Window.unboundedPreceding, Window.currentRow)))
    .withColumn("_cum_lo", F.coalesce(F.lag("_cum_hi", 1).over(win_faixa), F.lit(0)))
    .withColumn("excluidas_faixa",
        F.coalesce(
            F.least(
                F.expr("_cum_hi div total_exames_muni_ano") - F.expr("_cum_lo div total_exames_muni_ano"),
                F.col("total_exames_faixa")),
            F.lit(0)))
    .withColumn("total_exames_validos", F.col("total_exames_faixa") - F.col("excluidas_faixa"))
    .drop("_quota_num", "_cum_hi", "_cum_lo", "total_exames_muni_ano")
)

# Correção (31/out/2026): FULL OUTER JOIN entre numerador e denominador.
# O LEFT join anterior descartava faixas presentes no relatório de faixa etária
# mas sem coluna no relatório de alterados (TABNET omite colunas zeradas:
# '10 a 14' em 2024/25, 'Acima de 79' em 2025, 'Até 9 anos' e 'Ignorado'),
# deixando o denominador em 130.987. Com o full outer + coalesce(0) no numerador,
# a tabela cobre a grade completa de faixas por município/ano e o denominador
# fecha exatamente com o total de exames da regional (131.058).
df_q5 = (
    df_alterados.alias("a")
    .join(df_denominador.alias("d"), on=["codigo_municipio", "ano", "faixa_etaria"], how="full_outer")
    .select(
        F.col("codigo_municipio"),
        F.coalesce(F.col("a.municipio"), F.col("d.municipio")).alias("municipio"),
        F.col("ano"),
        F.col("faixa_etaria"),
        F.coalesce(F.col("a.quantidade_alterados"), F.lit(0)).alias("quantidade_alterados"),
        F.col("d.total_exames_faixa"),
        F.coalesce(F.col("d.qtd_excluidas_muni_ano"), F.lit(0)).alias("qtd_excluidas_muni_ano"),
        F.coalesce(F.col("d.excluidas_faixa"), F.lit(0)).alias("excluidas_faixa"),
        F.col("d.total_exames_validos"),
    )
    .withColumn(
        "proporcao_alterados_pct",
        F.when(F.col("total_exames_validos") > 0,
               F.round((F.col("quantidade_alterados") / F.col("total_exames_validos")) * 100, 2)),
    )
    # Agrupamento em 3 categorias (solicitado em 07/nov/2026): '<= 24 anos',
    # '25 a 64 anos' e '>= 65 anos'. 'Ignorado' permanece como própria categoria
    # (não é alocada em grupo etário). A granularidade da tabela NÃO muda —
    # faixa_etaria é preservada para as análises detalhadas da Seção 8; a
    # coluna grupo_etario_3 permite agregar a matriz territorial de proporção
    # de alterados em 3 categorias (taxa ponderada = soma ÷ soma por grupo).
    .withColumn(
        "grupo_etario_3",
        F.when(F.col("faixa_etaria").isin("Até 9 anos", "10 a 14 anos", "15 a 19 anos", "20 a 24 anos"), F.lit("<= 24 anos"))
         .when(F.col("faixa_etaria").isin("25 a 29 anos", "30 a 34 anos", "35 a 39 anos", "40 a 44 anos", "45 a 49 anos", "50 a 54 anos", "55 a 59 anos", "60 a 64 anos"), F.lit("25 a 64 anos"))
         .when(F.col("faixa_etaria").isin("65 a 69 anos", "70 a 74 anos", "75 a 79 anos", "Acima de 79 anos"), F.lit(">= 65 anos"))
         .otherwise(F.lit("Ignorado")),
    )
    .withColumn("cd_ibge", F.lpad(F.col("codigo_municipio").cast("string"), 6, "0"))
    .withColumn("_silver_source", F.lit(
        f"{CATALOG}.{SCHEMA_SILVER}.resultados_citopatologicos + {CATALOG}.{SCHEMA_SILVER}.faixa_etaria + {CATALOG}.{SCHEMA_SILVER}.adequabilidade"))
    .withColumn("_gold_processed_time", F.current_timestamp())
    .select(
        "codigo_municipio", "cd_ibge", "municipio", "ano", "faixa_etaria", "grupo_etario_3",
        "quantidade_alterados", "total_exames_faixa", "qtd_excluidas_muni_ano",
        "excluidas_faixa", "total_exames_validos", "proporcao_alterados_pct",
        "_silver_source", "_gold_processed_time",
    )
)

validar_e_gravar(
    df_q5,
    "fato_exames_alterados",
    pk_cols=["codigo_municipio", "ano", "faixa_etaria"],
    obrigatorio_cols=["codigo_municipio", "municipio", "ano", "faixa_etaria"],
)

display(df_q5.orderBy("codigo_municipio", "ano", "faixa_etaria").limit(20))

## 6. Validação Final das Tabelas Gold

Verificação pós-gravação lendo as tabelas diretamente do Unity Catalog:
* Contagem de linhas de cada tabela fato
* Soma de `participacao_territorial_pct` por competência ≈ 100 (Q1/Q2)
* Soma de `proporcao_pct` por município/ano ≈ 100 (Q3 e Q4)
* Soma das 3 proporções do KPI de população-alvo ≈ 100 (`fato_indicador_populacao_alvo`)
* Q5 — invariantes do denominador com exclusão de insatisfatórias/rejeitadas: alocação proporcional exata por município/ano (SUM das excluídas por faixa = total declarado) e fechamento global (total_faixa − excluídas = válidos = satisfatórias da Silver); nenhum denominador negativo ou menor que o numerador
* Escopo: 11 municípios da 10ª CRS, anos 2020–2025

In [0]:
# ---------------------------------------------------------------------------
# 6. Validação final — lê as tabelas Gold gravadas no Unity Catalog
# ---------------------------------------------------------------------------
tabelas_gold = [
    "fato_exames_temporais_territoriais",
    "fato_analise_outubro_rosa",
    "fato_conformidade_faixa_etaria",
    "fato_indicador_populacao_alvo",
    "fato_adequabilidade_amostras",
    "fato_exames_alterados",
]

print("=== Contagem de linhas (Gold no Unity Catalog) ===")
for t in tabelas_gold:
    df_t = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.{t}")
    print(f"  {t}: {df_t.count()} linhas")

# 1. Q1/Q2: participação territorial soma ~100 por competência
df_g1 = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.fato_exames_temporais_territoriais")
soma_part = (
    df_g1.groupBy("dt_competencia")
    .agg(F.round(F.sum("participacao_territorial_pct"), 1).alias("soma_participacao"))
)
err_part = soma_part.filter(~F.col("soma_participacao").between(99.5, 100.5)).count()
print(f"\nQ1/Q2 — competências fora de [99.5, 100.5]% de participação somada: {err_part}")

# 2. Q3: proporções somam ~100 por município/ano
df_g3 = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.fato_conformidade_faixa_etaria")
err_q3 = (
    df_g3.groupBy("codigo_municipio", "ano")
    .agg(F.round(F.sum("proporcao_pct"), 1).alias("s"))
    .filter(~F.col("s").between(99.5, 100.5))
    .count()
)
print(f"Q3 — pares município/ano fora de [99.5, 100.5]%: {err_q3}")

# 2b. Q3-KPI: proporções da população-alvo somam ~100 por município/ano
df_ind = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.fato_indicador_populacao_alvo")
err_ind = (
    df_ind.withColumn(
        "soma",
        F.round(
            F.col("proporcao_populacao_alvo_pct")
            + F.col("proporcao_sobrerastreamento_pct")
            + F.col("proporcao_subrastreamento_pct"), 1),
    )
    .filter(~F.col("soma").between(99.5, 100.5))
    .count()
)
print(f"Q3-KPI — município/ano fora de [99.5, 100.5]%: {err_ind}")

# 3. Q4: proporções somam ~100 por município/ano
df_g4 = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.fato_adequabilidade_amostras")
err_q4 = (
    df_g4.groupBy("codigo_municipio", "ano")
    .agg(F.round(F.sum("proporcao_adequabilidade_pct"), 1).alias("s"))
    .filter(~F.col("s").between(99.5, 100.5))
    .count()
)
print(f"Q4 — pares município/ano fora de [99.5, 100.5]%: {err_q4}")

# 3b. Q5 (03/nov/2026): invariantes do denominador com exclusão de insatisfatórias/rejeitadas
df_g5 = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.fato_exames_alterados")
df_adeq_s = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.adequabilidade")

# (a) Alocação proporcional exata: soma das exclusões alocadas por faixa == total declarado
err_q5_aloc = (
    df_g5.groupBy("codigo_municipio", "ano")
    .agg(F.sum("excluidas_faixa").alias("alocadas"), F.max("qtd_excluidas_muni_ano").alias("declaradas"))
    .filter(F.col("alocadas") != F.col("declaradas"))
    .count()
)
print(f"Q5 (a) — pares município/ano com alocação de exclusões divergente: {err_q5_aloc}")

# (b) Fechamento global: total_faixa − excluídas == válidos == satisfatórias da Silver
tot_q5 = df_g5.agg(
    F.sum("total_exames_faixa").alias("faixa"),
    F.sum("excluidas_faixa").alias("excluidas"),
    F.sum("total_exames_validos").alias("validos"),
    F.sum("quantidade_alterados").alias("alterados"),
).collect()[0]
sat_silver = (
    df_adeq_s.filter(F.col("categoria_adequabilidade") == "Satisfatoria")
    .agg(F.sum("quantidade")).collect()[0][0]
)
err_q5_fech = int((tot_q5["faixa"] - tot_q5["excluidas"]) != tot_q5["validos"]) + \
              int(tot_q5["validos"] != sat_silver)
print(f"Q5 (b) — total_exames_faixa ({tot_q5['faixa']:,}) − excluidas ({tot_q5['excluidas']:,}) "
      f"= validos ({tot_q5['validos']:,}) | satisfatórias Silver: {sat_silver:,} "
      f"| alterados: {tot_q5['alterados']:,} | divergências: {err_q5_fech}")

# (c) Sanidade: nenhum denominador negativo ou menor que o numerador
err_q5_neg = df_g5.filter(
    (F.col("total_exames_validos") < 0)
    | ((F.col("total_exames_validos").isNotNull())
       & (F.col("total_exames_validos") < F.col("quantidade_alterados")))
).count()
print(f"Q5 (c) — linhas com denominador negativo ou < alterados: {err_q5_neg}")

# 4. Escopo territorial e temporal
escopo = df_g1.agg(
    F.countDistinct("codigo_municipio").alias("municipios"),
    F.min("ano").alias("ano_min"),
    F.max("ano").alias("ano_max"),
)
display(escopo)

assert (err_part == 0 and err_q3 == 0 and err_ind == 0 and err_q4 == 0
        and err_q5_aloc == 0 and err_q5_fech == 0 and err_q5_neg == 0), \
    "❌ Validação final falhou — verifique as contagens acima."
print("\n✅ Validação final da Camada Gold concluída com sucesso.")

## 7. Análise Complementar — Sazonalidade Mensal (somente leitura)

**Objetivo:** perfil sazonal dos 12 meses agregando os 6 anos (2020–2025) a partir de `silver.exames_geral` (total 131.058 exames), para identificar os meses fracos e subsidiar recomendações de gestão (manutenção de equipamentos, férias, capacitação de coletadoras).

⚠️ **Nota metodológica:** ago/2022 está ausente na fonte TABNET — a série tem **71 ano-meses** (não 72); o índice de agosto é calculado sobre 5 ano-meses e os demais meses sobre 6. Esta seção é **somente leitura** sobre a Silver — não grava tabela Gold nem modifica tabelas.

In [0]:
%sql
-- 7.1 Perfil sazonal regional: índice sazonal por mês (100 = mês médio)
-- ago/2022 ausente na fonte TABNET: COUNT(*) reflete os ano-meses disponíveis (agosto = 5, demais meses = 6)
WITH mes_ano AS (
  SELECT ano, mes, SUM(quantidade) AS qtd
  FROM `tcc-analise-siscan-10crs`.silver.exames_geral
  GROUP BY ano, mes
),
base AS (
  SELECT AVG(qtd) AS media_geral FROM mes_ano
)
SELECT
  m.mes,
  CASE m.mes
    WHEN 1 THEN 'Janeiro' WHEN 2 THEN 'Fevereiro' WHEN 3 THEN 'Marco' WHEN 4 THEN 'Abril'
    WHEN 5 THEN 'Maio' WHEN 6 THEN 'Junho' WHEN 7 THEN 'Julho' WHEN 8 THEN 'Agosto'
    WHEN 9 THEN 'Setembro' WHEN 10 THEN 'Outubro' WHEN 11 THEN 'Novembro' WHEN 12 THEN 'Dezembro'
  END AS nome_mes,
  COUNT(*) AS ano_mes_disponiveis,
  ROUND(AVG(m.qtd), 1) AS media_mensal,
  ROUND(AVG(m.qtd) / b.media_geral * 100, 1) AS indice_sazonal,
  ROUND((AVG(m.qtd) / b.media_geral - 1) * 100, 1) AS desvio_pct_vs_media,
  RANK() OVER (ORDER BY AVG(m.qtd) ASC) AS rank_mais_fraco
FROM mes_ano m CROSS JOIN base b
GROUP BY m.mes, b.media_geral
ORDER BY m.mes

In [0]:
%sql
-- 7.2 Matriz mês × ano: consistência do padrão sazonal nos 6 anos
-- (ago/2022 ausente na fonte → NULL em a2022; mar-mai/2020 muito baixos por pandemia COVID)
WITH mes_ano AS (
  SELECT ano, mes, SUM(quantidade) AS qtd
  FROM `tcc-analise-siscan-10crs`.silver.exames_geral
  GROUP BY ano, mes
)
SELECT
  mes,
  MAX(CASE WHEN ano = 2020 THEN qtd END) AS a2020,
  MAX(CASE WHEN ano = 2021 THEN qtd END) AS a2021,
  MAX(CASE WHEN ano = 2022 THEN qtd END) AS a2022,
  MAX(CASE WHEN ano = 2023 THEN qtd END) AS a2023,
  MAX(CASE WHEN ano = 2024 THEN qtd END) AS a2024,
  MAX(CASE WHEN ano = 2025 THEN qtd END) AS a2025,
  ROUND(AVG(qtd), 1) AS media
FROM mes_ano
GROUP BY mes
ORDER BY mes

In [0]:
%sql
-- 7.3 Sazonalidade por município: índice relativo à média do PRÓPRIO município
-- (neutraliza diferenças de volume entre municípios e evidencia o padrão sazonal local)
WITH mun_mes AS (
  SELECT codigo_municipio, municipio, mes,
         SUM(quantidade) AS qtd_total,
         COUNT(DISTINCT ano) AS n_anos
  FROM `tcc-analise-siscan-10crs`.silver.exames_geral
  GROUP BY 1, 2, 3
),
mun_media AS (
  SELECT codigo_municipio, municipio, AVG(qtd_total / n_anos) AS media_mes_muni
  FROM mun_mes
  GROUP BY 1, 2
),
idx AS (
  SELECT m.codigo_municipio, m.municipio, m.mes,
         m.qtd_total / m.n_anos AS media_mensal,
         (m.qtd_total / m.n_anos) / g.media_mes_muni * 100 AS indice
  FROM mun_mes m
  JOIN mun_media g USING (codigo_municipio, municipio)
)
SELECT
  municipio,
  MAX(CASE WHEN rank_asc = 1 THEN CONCAT(nome, ' (', ROUND(indice, 0), ')') END) AS mes_mais_fraco,
  MAX(CASE WHEN mes IN (5, 6, 7, 8, 9) THEN ROUND(indice, 1) END) AS indice_maio_set,
  MAX(CASE WHEN mes IN (11, 12) THEN ROUND(indice, 1) END) AS indice_nov_dez,
  MAX(CASE WHEN rank_desc = 1 THEN CONCAT(nome, ' (', ROUND(indice, 0), ')') END) AS mes_mais_forte,
  ROUND(SUM(media_mensal), 0) AS media_anual
FROM (
  SELECT *,
    DENSE_RANK() OVER (PARTITION BY municipio ORDER BY indice ASC) AS rank_asc,
    DENSE_RANK() OVER (PARTITION BY municipio ORDER BY indice DESC) AS rank_desc,
    CASE mes
      WHEN 1 THEN 'Jan' WHEN 2 THEN 'Fev' WHEN 3 THEN 'Mar' WHEN 4 THEN 'Abr'
      WHEN 5 THEN 'Mai' WHEN 6 THEN 'Jun' WHEN 7 THEN 'Jul' WHEN 8 THEN 'Ago'
      WHEN 9 THEN 'Set' WHEN 10 THEN 'Out' WHEN 11 THEN 'Nov' WHEN 12 THEN 'Dez'
    END AS nome
  FROM idx
)
GROUP BY municipio
ORDER BY indice_nov_dez DESC

In [0]:
# ---------------------------------------------------------------------------
# 7.4 Sazonalidade — validação do total e visualização do índice sazonal
# (célula autônoma: reexecutável isolada, sem depender das células anteriores)
# ---------------------------------------------------------------------------
from pyspark.sql import functions as F
import matplotlib.pyplot as plt

CATALOG = "`tcc-analise-siscan-10crs`"
SCHEMA_SILVER = "silver"

# Validação global: o total do período deve fechar em 131.058
total = (
    spark.table(f"{CATALOG}.{SCHEMA_SILVER}.exames_geral")
    .agg(F.sum("quantidade").alias("total"))
    .collect()[0]["total"]
)
print(f"Total de exames 2020-2025: {total:,} (esperado: 131.058)")
assert int(total) == 131058, f"Total divergente do esperado: {total}"

# Perfil sazonal regional (mesma lógica da célula SQL 7.1) para o gráfico
df_sazon = (
    spark.table(f"{CATALOG}.{SCHEMA_SILVER}.exames_geral")
    .groupBy("ano", "mes")
    .agg(F.sum("quantidade").alias("qtd"))
)
media_geral = df_sazon.agg(F.avg("qtd").alias("mg")).collect()[0]["mg"]

df_sazon_plot = (
    df_sazon.groupBy("mes")
    .agg(F.round(F.avg("qtd") / F.lit(media_geral) * 100, 1).alias("indice_sazonal"))
    .withColumn(
        "nome_mes",
        F.expr("""
          CASE mes
            WHEN 1 THEN 'Janeiro' WHEN 2 THEN 'Fevereiro' WHEN 3 THEN 'Marco' WHEN 4 THEN 'Abril'
            WHEN 5 THEN 'Maio' WHEN 6 THEN 'Junho' WHEN 7 THEN 'Julho' WHEN 8 THEN 'Agosto'
            WHEN 9 THEN 'Setembro' WHEN 10 THEN 'Outubro' WHEN 11 THEN 'Novembro' WHEN 12 THEN 'Dezembro'
          END
        """),
    )
    .orderBy("mes")
    .toPandas()
)

fraca = df_sazon_plot.loc[df_sazon_plot["mes"].between(5, 9), "indice_sazonal"].mean()
pico = df_sazon_plot.loc[df_sazon_plot["mes"].isin([11, 12]), "indice_sazonal"].mean()
print(f"Janela fraca (mai-set): índice médio {fraca:.1f} | Pico (nov-dez): índice médio {pico:.1f} | razão ≈ {pico / fraca:.1f}x")

# Gráfico: índice sazonal por mês (vermelho = janela fraca mai-set; verde = pico nov-dez)
cores = ["#d9534f" if 5 <= m <= 9 else ("#5cb85c" if m in (11, 12) else "#adb5bd") for m in df_sazon_plot["mes"]]
fig, ax = plt.subplots(figsize=(11, 5))
barras = ax.bar(df_sazon_plot["nome_mes"], df_sazon_plot["indice_sazonal"], color=cores, edgecolor="black", linewidth=0.5)
ax.axhline(100, color="black", linestyle="--", linewidth=1, label="Mês médio (índice 100)")
for b, v in zip(barras, df_sazon_plot["indice_sazonal"]):
    ax.text(b.get_x() + b.get_width() / 2, v + 3, f"{v:.0f}", ha="center", fontsize=9)
ax.set_title("Índice sazonal mensal — exames citopatológicos, 10ª CRS (2020-2025)")
ax.set_ylabel("Índice (média mensal geral = 100)")
ax.set_ylim(0, 185)
ax.legend()
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

### 7.5 Conclusões da sazonalidade e recomendações de gestão

**Síntese do perfil sazonal** (índice 100 = mês médio; ago/2022 ausente na fonte TABNET):

| Janela | Meses | Índice sazonal |
| --- | --- | --- |
| Janela fraca | maio–setembro | 76,9–82,7 (≈17-23% abaixo da média) |
| Neutros | janeiro–abril, outubro | 85,6–108,8 |
| Pico | novembro–dezembro | 157,6–162,6 (≈2× a janela fraca em volume mensal) |

1. **Janela fraca: maio–setembro** — a produção cai ~17-23% abaixo do mês médio em toda a CRS. Recomendação: é a janela ótima para **manutenção de equipamentos, férias das equipes e capacitação de coletadoras**, com perda mínima de capacidade produtiva.
2. **Pico nov-dez uniforme nos 11 municípios** (índices ~135-212; nov ou dez é o mês mais forte em todos, com topo Itaqui 212, Quarai 208 e Sant'Ana 189) — fenômeno **sistêmico**, não local: compatível com busca de metas anuais e/ou artefato de digitação TABNET por data de processamento. **Limitação a declarar no TCC:** o pico (≈2.955 exames/mês) é cerca de 2× a janela fraca (≈1.475/mês) — não recomendar ampliação de capacidade com base no pico sem investigar a causa.
3. **Janeiro forte (108,5)** — atípico para serviços de saúde no início do ano e compatível com transbordo do processamento de dezembro, reforçando a hipótese de artefato de registro.
4. **Consistência temporal** — o padrão se repete nos 6 anos, inclusive fora da pandemia (mar-mai/2020 muito baixos por COVID); ver matriz mês × ano da célula 7.2.

## 8. Análise Exploratória — Exames Alterados (Q5, somente leitura)

> Migrada do notebook `04_analise_exames_alterados` em 03/nov/2026 (descontinuado) — as análises ficam consolidadas neste notebook (seções 7 e 8).

EDA sobre `tcc-analise-siscan-10crs.gold.fato_exames_alterados` (município × ano × faixa etária, 2020–2025): o denominador exclui amostras insatisfatórias (390) e rejeitadas (70), que não geram laudo diagnóstico definitivo, restando 130.598 exames analisáveis; 4.479 alterados.

**Perguntas norteadoras:**
1. Qual faixa etária concentra a maior taxa de exames alterados?
2. A taxa mudou ao longo do tempo (2020–2025)?
3. Quais municípios se destacam — e qual o efeito de bases pequenas (IC 95%)?
4. Existe relação entre volume de exames e taxa de alteração?
5. É possível "prever" algo? → modelo logístico binomial (prob. de alteração por faixa/ano) e projeção de tendência, com limitações explícitas.

**Regra metodológica:** taxas sempre pela **razão de somas** (Σ alterados ÷ Σ válidos), nunca média de percentuais; faixas `Ignorado` e `Até 9 anos` excluídas das taxas (base ínfima — mesma regra do dashboard AI/BI).

In [0]:
# ---------------------------------------------------------------------------
# 8.1 CARGA E VISÃO GERAL
# (migrado do notebook 04; .cache() removido — não suportado no Serverless,
#  que gerencia cache automaticamente; FAIXAS_EXCLUIR definida aqui para
#  viabilizar execução parcial das células 8.2–8.7)
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import functions as F

df = spark.table("`tcc-analise-siscan-10crs`.gold.fato_exames_alterados")

FAIXAS_EXCLUIR = ["Ignorado", "Até 9 anos"]  # base ínfima / fora da diretriz

n_rows = df.count()
tot = df.select(
    F.sum("total_exames_validos").alias("validos"),
    F.sum("quantidade_alterados").alias("alterados"),
).collect()[0]

print(f"Linhas (município × ano × faixa): {n_rows}")
print(f"Total de exames válidos: {tot['validos']:,}")
print(f"Total de exames alterados: {tot['alterados']:,}")
print(f"Taxa global ponderada: {tot['alterados'] / tot['validos'] * 100:.3f}%")
print("\nCobertura e volume por ano:")
df.select(
    F.countDistinct("municipio").alias("municipios"),
    F.countDistinct("ano").alias("anos"),
    F.countDistinct("faixa_etaria").alias("faixas"),
).show()
df.groupBy("ano").agg(
    F.sum("total_exames_validos").alias("validos"),
    F.sum("quantidade_alterados").alias("alterados"),
    F.round(F.sum("quantidade_alterados") / F.sum("total_exames_validos") * 100, 2).alias("taxa_pct"),
).orderBy("ano").show()

In [0]:
# ---------------------------------------------------------------------------
# 8.2 TAXA PONDERADA POR FAIXA ETÁRIA (razão de somas — nunca média de %)
# ---------------------------------------------------------------------------
taxa_faixa = (
    df.filter(~F.col("faixa_etaria").isin(FAIXAS_EXCLUIR))
    .groupBy("faixa_etaria")
    .agg(
        F.sum("quantidade_alterados").alias("alterados"),
        F.sum("total_exames_validos").alias("validos"),
    )
    .withColumn("taxa_pct", F.round(F.col("alterados") / F.col("validos") * 100, 2))
    .orderBy(F.desc("taxa_pct"))
)
display(taxa_faixa)

tf = taxa_faixa.toPandas()
top = tf.iloc[0]
print(f">>> Maior taxa de alteração: {top['faixa_etaria']} — {top['taxa_pct']}% "
      f"({top['alterados']:.0f} alterados de {top['validos']:.0f} exames)")

In [0]:
# ---------------------------------------------------------------------------
# 8.3 HEATMAP — TAXA (%) POR FAIXA × ANO
# ---------------------------------------------------------------------------
pivot = (
    df.filter(~F.col("faixa_etaria").isin(FAIXAS_EXCLUIR))
    .groupBy("faixa_etaria", "ano")
    .agg(F.round(F.sum("quantidade_alterados") / F.sum("total_exames_validos") * 100, 2).alias("taxa"))
    .toPandas()
    .pivot(index="faixa_etaria", columns="ano", values="taxa")
)

ordem = ["10 a 14 anos", "15 a 19 anos", "20 a 24 anos", "25 a 29 anos", "30 a 34 anos",
         "35 a 39 anos", "40 a 44 anos", "45 a 49 anos", "50 a 54 anos", "55 a 59 anos",
         "60 a 64 anos", "65 a 69 anos", "70 a 74 anos", "75 a 79 anos", "Acima de 79 anos"]
pivot = pivot.reindex([f for f in ordem if f in pivot.index])

fig, ax = plt.subplots(figsize=(11, 6))
im = ax.imshow(pivot.values.astype(float), cmap="OrRd", aspect="auto")
ax.set_xticks(range(len(pivot.columns)), [str(int(c)) for c in pivot.columns])
ax.set_yticks(range(len(pivot.index)), pivot.index)
ax.tick_params(labelsize=8)
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        v = pivot.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, label="Taxa de alterados (%)")
ax.set_title("Taxa de exames alterados por faixa etária e ano — 10ª CRS (2020–2025)")
plt.tight_layout()
plt.show()

In [0]:
# ---------------------------------------------------------------------------
# 8.4 EVOLUÇÃO ANUAL DA TAXA PONDERADA (tendência linear simples)
# ---------------------------------------------------------------------------
anual = (
    df.groupBy("ano")
    .agg(F.sum("quantidade_alterados").alias("alt"), F.sum("total_exames_validos").alias("val"))
    .orderBy("ano")
    .toPandas()
)
anual["taxa"] = anual.alt / anual.val * 100

b1, b0 = np.polyfit(anual.ano, anual.taxa, 1)  # MQO sobre 6 pontos anuais
print(anual[["ano", "alt", "val", "taxa"]].round(3).to_string(index=False))
print(f"\nTendência linear: {b1:+.3f} pontos percentuais por ano")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(anual.ano, anual.taxa, marker="o", label="Taxa observada")
ax.plot(anual.ano, b0 + b1 * anual.ano, "--", label="Tendência linear")
for _, r in anual.iterrows():
    ax.annotate(f"{r.taxa:.2f}%", (r.ano, r.taxa), textcoords="offset points", xytext=(0, 7), fontsize=8)
ax.set_xlabel("Ano"); ax.set_ylabel("Taxa (%)")
ax.set_title("Taxa anual de exames alterados — 10ª CRS")
ax.legend()
plt.tight_layout(); plt.show()

In [0]:
# ---------------------------------------------------------------------------
# 8.5 MUNICÍPIOS — TAXA PONDERADA, IC 95% DE WILSON E CORRELAÇÃO VOLUME × TAXA
# ---------------------------------------------------------------------------
mun = (
    df.filter(~F.col("faixa_etaria").isin(FAIXAS_EXCLUIR))
    .groupBy("municipio")
    .agg(F.sum("quantidade_alterados").alias("alt"), F.sum("total_exames_validos").alias("val"))
    .toPandas()
)
mun["taxa"] = mun.alt / mun.val * 100


def wilson(k, n, z=1.96):
    """Intervalo de confiança 95% para proporção — apropriado para bases pequenas."""
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (c - h) * 100, (c + h) * 100


# result_type="expand": substitui o apply que retornava pd.Series (deprecado no pandas >= 2.1)
mun[["ic_lo", "ic_hi"]] = mun.apply(lambda r: wilson(r["alt"], r["val"]), axis=1, result_type="expand")
mun = mun.sort_values("taxa", ascending=False).reset_index(drop=True)
print(mun.round(2).to_string(index=False))

# Correlação de Spearman: municípios com mais exames têm taxa maior/menor?
rk_v, rk_t = mun["val"].rank(), mun["taxa"].rank()
rho = np.corrcoef(rk_v, rk_t)[0, 1]
print(f"\nSpearman volume × taxa (municípios): rho = {rho:+.2f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(mun.taxa, range(len(mun)),
            xerr=[mun.taxa - mun.ic_lo, mun.ic_hi - mun.taxa],
            fmt="o", capsize=3)
ax.set_yticks(range(len(mun)), mun.municipio, fontsize=8)
ax.axvline(tot["alterados"] / tot["validos"] * 100, color="gray", ls=":",
           label=f"Taxa regional {tot['alterados'] / tot['validos'] * 100:.2f}%")
ax.set_xlabel("Taxa (%) com IC 95% de Wilson")
ax.set_title("Taxa de alterados por município (2020–2025)")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [0]:
# ---------------------------------------------------------------------------
# 8.6 POPULAÇÃO-ALVO (25–64) vs SOBRE/SUB-RASTREAMENTO — TAXAS PONDERADAS
# ---------------------------------------------------------------------------
def grupo(faixa: str) -> str:
    if any(faixa.startswith(p) for p in ["25 a 29", "30 a 34", "35 a 39", "40 a 44",
                                          "45 a 49", "50 a 54", "55 a 59", "60 a 64"]):
        return "25–64 (população-alvo)"
    if faixa.startswith(("10 a 14", "15 a 19", "20 a 24")):
        return "<25 (sobrerastreamento)"
    return "65+ (sub-rastreamento)"


g = df.filter(~F.col("faixa_etaria").isin(FAIXAS_EXCLUIR)).toPandas()
g["grupo"] = g["faixa_etaria"].map(grupo)
res = g.groupby("grupo").agg(alt=("quantidade_alterados", "sum"),
                              val=("total_exames_validos", "sum"))
res["taxa_pct"] = (res.alt / res.val * 100).round(2)
print(res.sort_values("taxa_pct", ascending=False).to_string())
print(f"\nSoma das taxas ≠ 100% pois a taxa é de ALTERADOS (não a distribuição de exames).")

In [0]:
# ---------------------------------------------------------------------------
# 8.7 MODELO LOGÍSTICO BINOMIAL — P(exame alterado) ~ idade + ano
# (dados agrupados; IRLS em numpy — sem dependências externas:
#  scikit-learn/statsmodels não garantidos no serverless)
# ---------------------------------------------------------------------------
IDADE = {"10 a 14 anos": 12.5, "15 a 19 anos": 17.5, "20 a 24 anos": 22.5, "25 a 29 anos": 27.5,
         "30 a 34 anos": 32.5, "35 a 39 anos": 37.5, "40 a 44 anos": 42.5, "45 a 49 anos": 47.5,
         "50 a 54 anos": 52.5, "55 a 59 anos": 57.5, "60 a 64 anos": 62.5, "65 a 69 anos": 67.5,
         "70 a 74 anos": 72.5, "75 a 79 anos": 77.5, "Acima de 79 anos": 82.5}

m = df.filter(~F.col("faixa_etaria").isin(FAIXAS_EXCLUIR)).toPandas()
m["idade"] = m["faixa_etaria"].map(IDADE)
m = m.dropna(subset=["idade"]).copy()
# Linhas com denominador ausente/zero quebrariam o binomial (k/n): são faixas que
# existem só no relatório de alterados (full outer da gold, ex. rótulos não informados).
n_antes = len(m)
m = m[m["total_exames_validos"].notna() & (m["total_exames_validos"] > 0)
      & m["quantidade_alterados"].notna()].copy()
print(f"Linhas usadas no modelo: {len(m)} (descartadas {n_antes - len(m)} com denominador NULL/0)")


def fit_binomial(X, k, n, iters=200, tol=1e-10):
    """Regressão logística ponderada sobre proporções (IRLS).
    Retorna beta, covariância e flag de convergência."""
    beta = np.zeros(X.shape[1])
    converged = False
    for _ in range(iters):
        eta = np.clip(X @ beta, -30, 30)
        mu = 1 / (1 + np.exp(-eta))
        w = n * mu * (1 - mu) + 1e-9
        z = eta + (k / n - mu) / np.maximum(mu * (1 - mu), 1e-9)
        beta_new = np.linalg.solve(X.T @ (w[:, None] * X), X.T @ (w * z))
        if np.max(np.abs(beta_new - beta)) < tol:
            beta = beta_new
            converged = True
            break
        beta = beta_new
    eta = np.clip(X @ beta, -30, 30)
    mu = 1 / (1 + np.exp(-eta))
    w = n * mu * (1 - mu) + 1e-9
    cov = np.linalg.inv(X.T @ (w[:, None] * X))
    return beta, cov, converged


idade_c = (m["idade"] - 47.5) / 10   # idade centralizada: OR por década a partir de 47,5 anos
ano_c = m["ano"] - 2022.5            # ano centralizado na série
X = np.column_stack([np.ones(len(m)), idade_c, ano_c])
beta, cov, converged = fit_binomial(X, m["quantidade_alterados"].astype(float).values,
                                     m["total_exames_validos"].astype(float).values)
assert converged, "❌ IRLS não convergiu em 200 iterações — coeficientes não confiáveis."
print("✅ IRLS convergiu (tol 1e-10).")

se = np.sqrt(np.diag(cov))
names = ["intercepto (taxa aos 47,5 anos, ano central)", "idade (por década)", "ano (tendência)"]
print("Coeficientes (log-odds) e razão de chances (OR) com IC 95%:")
for nm, b, s in zip(names, beta, se):
    lo, hi = np.exp(b - 1.96 * s), np.exp(b + 1.96 * s)
    print(f"  {nm:<42} b={b:+.4f} (ep={s:.4f})   OR={np.exp(b):.3f}  IC95%[{lo:.3f}; {hi:.3f}]")

# Probabilidade prevista por faixa (padrão etário, no ano central 2022.5)
faixas_unicas = sorted(m["faixa_etaria"].unique(), key=lambda f: IDADE[f])
pred = pd.DataFrame({"faixa": faixas_unicas, "idade": [IDADE[f] for f in faixas_unicas]})
xx = np.column_stack([np.ones(len(pred)), (pred["idade"] - 47.5) / 10, np.zeros(len(pred))])
pred["p_prevista_pct"] = 100 / (1 + np.exp(-(xx @ beta)))
obs = taxa_faixa.toPandas().set_index("faixa_etaria")["taxa_pct"]
pred["p_observada_pct"] = pred["faixa"].map(obs)
print("\nProbabilidade prevista vs observada por faixa:")
print(pred.round(2).to_string(index=False))

In [0]:
# ---------------------------------------------------------------------------
# 8.8 PROJEÇÃO ILUSTRATIVA PARA 2026 (efeito ano do modelo, método delta)
# ---------------------------------------------------------------------------
for alvo in (2026,):
    x = np.array([1.0, 0.0, alvo - 2022.5])   # idade na média; efeito do ano
    p = 1 / (1 + np.exp(-(x @ beta)))
    var = float(x @ cov @ x)
    se_p = p * (1 - p) * np.sqrt(var)
    print(f"Taxa prevista para {alvo}: {p * 100:.2f}%  "
          f"(IC95% delta ≈ [{(p - 1.96 * se_p) * 100:.2f}%; {(p + 1.96 * se_p) * 100:.2f}%])")

print("\n⚠️ Extrapolação com apenas 6 pontos anuais e dados AGREGADOS (sem variáveis individuais).")
print("   Usar como indicador de TENDÊNCIA para planejamento, não como previsão clínica.")

In [0]:
# ---------------------------------------------------------------------------
# 8.9 EXAMES ALTERADOS POR MUNICÍPIO × ANO (tabela dinâmica)
# (a conferência de integridade gold × silver correspondente migrou para o
#  notebook 05_auditoria_meses_silver, seção 4)
# ---------------------------------------------------------------------------
q5_muni_ano = (
    spark.table("`tcc-analise-siscan-10crs`.gold.fato_exames_alterados")
    .groupBy("municipio", "ano")
    .agg(F.sum("quantidade_alterados").alias("alterados"))
)

pivot_muni_ano = (
    q5_muni_ano.groupBy("municipio")
    .pivot("ano", [2020, 2021, 2022, 2023, 2024, 2025])
    .agg(F.sum("alterados"))
    .toPandas()
    .set_index("municipio")
)
pivot_muni_ano["Total"] = pivot_muni_ano.sum(axis=1)
pivot_muni_ano.loc["Total"] = pivot_muni_ano.sum()
print("Exames alterados por município e ano (10ª CRS, 2020–2025):")
print(pivot_muni_ano.fillna(0).astype(int).to_string())

### 8.10 Síntese — o que se pode extrair desta tabela

**Achados (reexecução de 03/nov/2026 — denominador = 130.598 exames analisáveis, excluindo 390 insatisfatórias e 70 rejeitadas):**
* Taxa global: **3,43%** (4.479 alterados / 130.598 exames analisáveis).
* Faixa com maior taxa: **10-14 anos (4,91%)** — porém base minúscula (8/163). Entre faixas robustas: **15-19 (4,40%)** e **35-39 (4,14%)**; na população-alvo, **25-29 (3,92%)** lidera.
* Padrão etário: a taxa **decresce com a idade** — modelo binomial: OR = **0,889 por década** de idade (IC95% 0,870–0,907), ou ~11% menos chance de resultado alterado a cada década.
* Evolução anual: 3,09% (2020) → pico **4,72% (2023)** → queda a **1,74% (2025)**; tendência −0,22 p.p./ano; efeito ano no modelo: OR = 0,927 (−7% ao ano).
* Grupos da diretriz: **<25 = 4,01%** > 25–64 = 3,43% > 65+ = 2,55% — o sobrerastreamento concentra a maior taxa, reforçando a diretriz de não rastrear <25 anos.
* Municípios: **Quaraí (4,60%)** e **Uruguaiana (4,55%)** no topo; **Sant'Ana do Livramento (1,69%)** e **São Gabriel (2,35%)** na base. IC de Wilson mostra que Maçambará (IC 1,88–6,62) e Santa Margarida (3,10–5,95) são estatisticamente imprecisos (poucos exames).
* Correlação volume × taxa (Spearman): **rho = −0,12** — não há relação entre volume de exames e taxa de alteração.
* Projeção ilustrativa 2026 (efeito ano): **2,53%** (IC95% 2,36–2,71).

### O que a tabela PERMITE
* Taxa de alteração ponderada por **faixa etária, ano e município** (razão de somas, com IC de Wilson para bases pequenas).
* **Padrão etário** do resultado alterado (via modelo logístico binomial: OR por década de idade).
* **Tendência temporal** 2020–2025 e projeção ilustrativa de taxa para 2026.
* **Ranking de municípios** com comparação estatisticamente honesta (IC 95%).

### O que a tabela NÃO permite
* **Motivo da alteração (verificado até a bronze em 03/nov/2026):** `bronze.resultados_raw` está no formato largo original do TABNET com colunas apenas de **município + 15 faixas etárias + Total**; os arquivos-fonte são unicamente os 6 CSVs `q5-siscan_colo_alterados_faixa_etaria_*.csv` (2020–2025); e o volume de origem (`siscan_files_bronze/dataset`) contém apenas as 4 famílias de relatórios já ingeridas (Q1/Q2 exames geral, Q3 faixa etária, Q4 adequabilidade, Q5 alterados) — **nenhum relatório com dimensão de diagnóstico**. Não há quebra por ASC-US/LSIL/HSIL/carcinoma em nenhuma camada, bronze incluída. Para decompor o “alterado” seria necessário exportar do SISCAN/TABNET o relatório de resultados por diagnóstico e ingeri-lo no pipeline bronze→silver→gold.
* Distribuição etária **exata** das amostras insatisfatórias/rejeitadas — o SISCAN não cruza adequabilidade × faixa etária, então as exclusões do denominador são alocadas proporcionalmente ao volume de exames de cada faixa (pressuposto metodológico declarável).
* Prever resultado de um **exame individual** (dados agregados — falácia ecológica).
* Ajustar por fatores de confusão individuais (história reprodutiva, HPV, etc.).
* Diagnóstico de causa — alterado ≠ necessariamente lesão (inclui atipias benignas/ASC-US).

> **Conferência de integridade:** o numerador da gold × silver (município/ano) é validado no notebook `05_auditoria_meses_silver` (seção 4).